# Retail Sales Analysis Template

Complete workflow for analyzing retail sales data:
- Data profiling
- KPI calculation
- Product performance
- Seasonal trends
- Customer segmentation
- Sales forecasting

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import sys
sys.path.append('../src')
from retail_utils import RetailAnalytics, RetailDataValidator

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load & Profile Data

In [ ]:
# Load your retail data
df = pd.read_csv('../data/raw/sales_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

## 2. Data Validation

In [ ]:
# Define required columns for your data
required_cols = ['date', 'sales_amount', 'quantity', 'product_id', 'customer_id']

validator = RetailDataValidator()
validation = validator.validate_sales_data(df, required_cols)

print(f"Valid: {validation['is_valid']}")
print(f"Rows: {validation['row_count']}, Columns: {validation['col_count']}")
if validation['issues']:
    print(f"\nIssues found:")
    for issue in validation['issues']:
        print(f"  - {issue}")

## 3. Calculate Key KPIs

In [ ]:
analytics = RetailAnalytics()
kpis = analytics.calculate_kpis(df, 'sales_amount', 'quantity')

print("Sales KPIs:")
for key, value in kpis.items():
    if isinstance(value, float):
        print(f"  {key}: ${value:,.2f}")
    else:
        print(f"  {key}: {value:,}")

## 4. Top/Bottom Products

In [ ]:
top_products = analytics.product_performance(df, 'product_id', 'sales_amount', 'quantity', top_n=10)
print("Top 10 Products by Sales:")
print(top_products)

# Visualize
top_products[('sales_amount', 'sum')].plot(kind='barh')
plt.xlabel('Total Sales')
plt.title('Top 10 Products by Revenue')
plt.tight_layout()
plt.savefig('../results/top_products.png', dpi=300)
plt.show()

## 5. Customer Segmentation (RFM)

In [ ]:
# Recency, Frequency, Monetary
df['date'] = pd.to_datetime(df['date'])
latest_date = df['date'].max()

rfm = df.groupby('customer_id').agg({
    'date': lambda x: (latest_date - x.max()).days,  # Recency
    'customer_id': 'count',  # Frequency
    'sales_amount': 'sum'  # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
print("RFM Summary:")
print(rfm.describe())

# Create segments
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1], duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4], duplicates='drop')
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4], duplicates='drop')

rfm['segment'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
print(f"\nCustomer Segments Distribution:\n{rfm['segment'].value_counts()}")

## 6. Seasonal Analysis

In [ ]:
seasonal = analytics.seasonal_analysis(df, 'date', 'sales_amount')
print("Seasonal Sales Trends:")
print(seasonal)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
seasonal['sum'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Monthly Sales Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/seasonal_trend.png', dpi=300)
plt.show()

## 7. Sales Forecast

In [ ]:
# Prepare daily sales
daily_sales = df.groupby('date')['sales_amount'].sum().reset_index()
daily_sales.set_index('date', inplace=True)

# Forecast next 30 days using different methods
forecast_avg = analytics.forecast_simple(daily_sales, 'sales_amount', periods=30, method='average')
forecast_trend = analytics.forecast_simple(daily_sales, 'sales_amount', periods=30, method='trend')

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
daily_sales['sales_amount'].plot(ax=ax, label='Historical', linewidth=2)
ax.plot(range(len(daily_sales), len(daily_sales) + 30), forecast_trend, 
        label='Forecast (Trend)', linestyle='--', linewidth=2)
ax.set_title('30-Day Sales Forecast')
ax.set_xlabel('Date')
ax.set_ylabel('Sales Amount')
ax.legend()
plt.tight_layout()
plt.savefig('../results/sales_forecast.png', dpi=300)
plt.show()

print(f"\nForecast Summary:")
print(f"  Average forecast (next 30 days): ${forecast_avg.mean():,.2f}/day")
print(f"  Trend forecast (next 30 days): ${forecast_trend.mean():,.2f}/day")

## 8. Export Results

In [ ]:
# Save processed data and results
rfm.to_csv('../results/customer_rfm.csv')
seasonal.to_csv('../results/seasonal_analysis.csv')

print("Results saved to ../results/")